# ⚡ Z-Fooocus
**Multi-Model Studio** — Z-Image Turbo · FLUX.2-klein · Qwen-Image-Edit

Generate · Img2Img · Inpaint with model switching

In [ ]:
# ── Step 1: Install & Download Models ─────────────────────
import os

# ── ComfyUI ───────────────────────────────────────────────
if not os.path.exists('/content/ComfyUI'):
    !git clone https://github.com/comfyanonymous/ComfyUI.git /content/ComfyUI
    %cd /content/ComfyUI
    !pip install -q -r requirements.txt

# ComfyUI-GGUF custom nodes
if not os.path.exists('/content/ComfyUI/custom_nodes/ComfyUI-GGUF'):
    !git clone https://github.com/city96/ComfyUI-GGUF.git /content/ComfyUI/custom_nodes/ComfyUI-GGUF
    !pip install -q -r /content/ComfyUI/custom_nodes/ComfyUI-GGUF/requirements.txt

!pip install -q rembg[gpu] onnxruntime-gpu
!pip install -q aria2

# ── Model dirs ────────────────────────────────────────────
DIFF    = '/content/ComfyUI/models/diffusion_models'
UNET    = '/content/ComfyUI/models/unet'
CLIP    = '/content/ComfyUI/models/clip'
TXTENC  = '/content/ComfyUI/models/text_encoders'
VAE     = '/content/ComfyUI/models/vae'
for d in [DIFF, UNET, CLIP, TXTENC, VAE]:
    os.makedirs(d, exist_ok=True)

# ── Z-Image Turbo ─────────────────────────────────────────
print('\n📦 Z-Image Turbo FP8')
if not os.path.exists(f'{DIFF}/z-image-turbo-fp8-e4m3fn.safetensors'):
    !aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/T5B/Z-Image-Turbo-FP8/resolve/main/z-image-turbo-fp8-e4m3fn.safetensors -d {DIFF} -o z-image-turbo-fp8-e4m3fn.safetensors
else:
    print('  ✅ z-image-turbo-fp8-e4m3fn.safetensors')

if not os.path.exists(f'{CLIP}/qwen_3_4b.safetensors'):
    !aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/Comfy-Org/z_image_turbo/resolve/main/split_files/text_encoders/qwen_3_4b.safetensors -d {CLIP} -o qwen_3_4b.safetensors
else:
    print('  ✅ qwen_3_4b.safetensors')

if not os.path.exists(f'{VAE}/ae.safetensors'):
    !aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/Comfy-Org/z_image_turbo/resolve/main/split_files/vae/ae.safetensors -d {VAE} -o ae.safetensors
else:
    print('  ✅ ae.safetensors')

# ── FLUX.2-klein FP8 ──────────────────────────────────────
print('\n📦 FLUX.2-klein 9B FP8')
if not os.path.exists(f'{UNET}/flux-2-klein-9b-kv-fp8.safetensors'):
    !aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/black-forest-labs/FLUX.2-klein-9b-kv-fp8/resolve/main/flux-2-klein-9b-kv-fp8.safetensors -d {UNET} -o flux-2-klein-9b-kv-fp8.safetensors
else:
    print('  ✅ flux-2-klein-9b-kv-fp8.safetensors')

# Qwen3 8B FP4 mixed — goes to BOTH clip/ AND text_encoders/ (ComfyUI searches both)
if not os.path.exists(f'{TXTENC}/qwen_3_8b_fp4mixed.safetensors'):
    !aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/Comfy-Org/vae-text-encorder-for-flux-klein-9b/resolve/main/split_files/text_encoders/qwen_3_8b_fp4mixed.safetensors -d {TXTENC} -o qwen_3_8b_fp4mixed.safetensors
else:
    print('  ✅ qwen_3_8b_fp4mixed.safetensors')

# FLUX.2 VAE
if not os.path.exists(f'{VAE}/flux2-vae.safetensors'):
    !aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/Comfy-Org/vae-text-encorder-for-flux-klein-9b/resolve/main/split_files/vae/flux2-vae.safetensors -d {VAE} -o flux2-vae.safetensors
else:
    print('  ✅ flux2-vae.safetensors')

# ── Qwen-Image-Edit-2511 GGUF Q4_K_M ──────────────────────
print('\n📦 Qwen-Image-Edit-2511 GGUF Q4_K_M')
if not os.path.exists(f'{UNET}/qwen-image-edit-2511-Q4_K_M.gguf'):
    !aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/unsloth/Qwen-Image-Edit-2511-GGUF/resolve/main/qwen-image-edit-2511-Q4_K_M.gguf -d {UNET} -o qwen-image-edit-2511-Q4_K_M.gguf
else:
    print('  ✅ qwen-image-edit-2511-Q4_K_M.gguf')

if not os.path.exists(f'{CLIP}/Qwen2.5-VL-7B-Instruct-UD-Q4_K_XL.gguf'):
    !aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/unsloth/Qwen2.5-VL-7B-Instruct-GGUF/resolve/main/Qwen2.5-VL-7B-Instruct-UD-Q4_K_XL.gguf -d {CLIP} -o Qwen2.5-VL-7B-Instruct-UD-Q4_K_XL.gguf
else:
    print('  ✅ Qwen2.5-VL-7B-Instruct-UD-Q4_K_XL.gguf')

if not os.path.exists(f'{VAE}/qwen_image_vae.safetensors'):
    !aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/Qwen/Qwen-Image-Edit-2511/resolve/main/vae/diffusion_pytorch_model.safetensors -d {VAE} -o qwen_image_vae.safetensors
else:
    print('  ✅ qwen_image_vae.safetensors')

# Symlink clip files to text_encoders (ComfyUI CLIPLoader searches text_encoders/)
import glob
for f in glob.glob(f'{CLIP}/*'):
    link = os.path.join(TXTENC, os.path.basename(f))
    if not os.path.exists(link):
        os.symlink(f, link)

print('\n🎉 All models ready!')

In [ ]:
# ── Step 2: Launch Z-Fooocus ──────────────────────────────
import os
os.chdir('/content')
REPO = 'https://github.com/MuntasirMalek/Z-Fooocus.git'
if os.path.exists('/content/Z-Fooocus'):
    %cd /content/Z-Fooocus
    !git pull --ff-only || true
else:
    !git clone {REPO}
    %cd /content/Z-Fooocus
!python app.py